# PRIM — When Does Pooling Reduce T1 Mortality?

For each of the 8 non-strict policies (3×3 grid minus baseline),
compares `death_rate_T1_BCAS` against the strict baseline (0,0)
and runs PRIM to discover the uncertainty sub-space where that
specific pooling policy **reduces** T1 mortality.

**Requires** a results.csv from an experiment with 3-level FF lever
variation over `facility_eligibility` and `ambulance_eligibility`.

In [ ]:
import sys
from pathlib import Path
_root = Path(".").resolve()
while not (_root / "paths.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))
import paths

In [ ]:
FOLDER_NAME = FOLDER_NAME
LEVER_COLS  = ["facility_eligibility", "ambulance_eligibility"]
OUTCOME     = "death_rate_T1_BCAS"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ema_workbench.analysis import prim

run_folder = paths.MULTI_RUN_OUTPUT / FOLDER_NAME

df = pd.read_csv(run_folder / "results.csv")
print(f"Loaded {len(df)} runs  |  {df['scenario'].nunique()} scenarios  |  "
      f"{df['policy'].nunique()} distinct policies")
for lev in LEVER_COLS:
    print(f"  {lev}: {sorted(df[lev].unique())}")

In [ ]:
meta_cols      = ["scenario", "policy", "model"]
meta_start     = min(df.columns.get_loc(c) for c in meta_cols if c in df.columns)
input_cols     = df.columns[:meta_start].tolist()
uncertainty_cols = [c for c in input_cols if c not in LEVER_COLS]

print(f"Uncertainties ({len(uncertainty_cols)}): {uncertainty_cols}")
print(f"Levers ({len(LEVER_COLS)}): {LEVER_COLS}")

baseline_mask = (df["facility_eligibility"] == 0) & (df["ambulance_eligibility"] == 0)
baseline_outcome = (
    df.loc[baseline_mask]
    .groupby("scenario")[OUTCOME]
    .mean()
    .rename("strict")
)

x_all = (
    df.loc[baseline_mask]
    .drop_duplicates(subset="scenario")
    .set_index("scenario")
    [uncertainty_cols]
    .astype(float)
)

# All non-strict policies
POLICIES = [
    (fe, ae)
    for fe in sorted(df["facility_eligibility"].unique())
    for ae in sorted(df["ambulance_eligibility"].unique())
    if not (fe == 0 and ae == 0)
]
POLICY_LABELS = {0: "strict", 1: "bn_pool", 2: "bde_pool"}
print(f"\nNon-strict policies: {len(POLICIES)}")
for fe, ae in POLICIES:
    print(f"  fac={POLICY_LABELS[fe]}({fe}), amb={POLICY_LABELS[ae]}({ae})")

## Per-Policy PRIM: Under Which Conditions Does This Policy Reduce T1 Mortality?

For each non-strict policy, computes `delta = strict - policy` for `death_rate_T1_BCAS`.
Positive delta = pooling reduced mortality (good). Runs PRIM on the binary
indicator `delta > 0`.

In [ ]:
results_summary = []

for fe, ae in POLICIES:
    label = f"fac={POLICY_LABELS[fe]}, amb={POLICY_LABELS[ae]}"
    mask = (df["facility_eligibility"] == fe) & (df["ambulance_eligibility"] == ae)
    policy_outcome = (
        df.loc[mask]
        .groupby("scenario")[OUTCOME]
        .mean()
        .rename("policy")
    )

    comp = pd.concat([baseline_outcome, policy_outcome], axis=1).dropna()
    comp["delta"] = comp["strict"] - comp["policy"]
    comp["pooling_reduces_mortality"] = (comp["delta"] > 0).astype(int)

    n_better = comp["pooling_reduces_mortality"].sum()
    n_total  = len(comp)
    results_summary.append({
        "policy": label,
        "n_better": n_better,
        "n_total": n_total,
        "pct": 100 * n_better / n_total,
        "mean_delta": comp["delta"].mean(),
    })

    print(f"\n{'='*70}")
    print(f"  {label}  —  pooling reduces T1 mortality in "
          f"{n_better}/{n_total} scenarios ({100*n_better/n_total:.1f}%)")
    print(f"  mean delta (strict − policy): {comp['delta'].mean():.4f}")
    print(f"={'='*70}")

    # --- Histograms ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(comp["delta"], bins=30, edgecolor="black")
    axes[0].axvline(0, color="red", ls="--")
    axes[0].set_xlabel(f"Delta ({OUTCOME}: strict − policy)")
    axes[0].set_ylabel("Scenarios")
    axes[0].set_title(f"{label}: delta distribution")

    axes[1].scatter(comp["strict"], comp["policy"],
                    c=comp["pooling_reduces_mortality"], cmap="RdYlGn_r",
                    edgecolors="black", alpha=0.7, s=30)
    lims = [min(comp[["strict","policy"]].min()),
            max(comp[["strict","policy"]].max())]
    axes[1].plot(lims, lims, "k--", alpha=0.4)
    axes[1].set_xlabel(f"{OUTCOME} (strict)")
    axes[1].set_ylabel(f"{OUTCOME} (policy)")
    axes[1].set_title(f"Below diagonal = pooling reduces mortality")
    plt.suptitle(label, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # --- PRIM ---
    if n_better < 3 or n_better > n_total - 3:
        print(f"  Too few cases ({n_better}) or too uniform — skipping PRIM.\n")
        continue

    x_prim = x_all.loc[comp.index].reset_index(drop=True)
    y_prim = comp["pooling_reduces_mortality"].values

    p = prim.Prim(x_prim, y_prim, threshold=0.5, threshold_type=prim.ABOVE)
    box = p.find_box()

    traj = box.peeling_trajectory
    score = traj["coverage"] * traj["density"]
    bi = int(score.idxmax())
    print(f"  Auto-selected box {bi}  "
          f"(coverage={traj.loc[bi, 'coverage']:.2f}, "
          f"density={traj.loc[bi, 'density']:.2f})")

    box.show_tradeoff()
    plt.title(f"PRIM tradeoff — {label}")
    plt.show()

    box.inspect(bi, style="graph")
    plt.suptitle(f"{label} — box {bi}", y=1.02)
    plt.tight_layout()
    plt.show()

    box.select(bi)
    lim = box.box_lim
    any_restricted = False
    for col in uncertainty_cols:
        lo, hi = lim[col].iloc[0], lim[col].iloc[1]
        full_lo, full_hi = x_prim[col].min(), x_prim[col].max()
        if lo > full_lo + 1e-9 or hi < full_hi - 1e-9:
            print(f"    {col:30s}  [{lo:.2f}, {hi:.2f}]  "
                  f"(full: [{full_lo:.2f}, {full_hi:.2f}])")
            any_restricted = True
    if not any_restricted:
        print("    No dimensions restricted.")
    print()

In [ ]:
summary_df = pd.DataFrame(results_summary)
print("\nSummary: fraction of scenarios where pooling reduces T1 mortality\n")
display(summary_df.style.format({"pct": "{:.1f}%", "mean_delta": "{:.4f}"}))